# 13 · Preemption and offload
Level L3 · Part 2 · core · ~25 min · builds on 12

## Goal
By the end of this lesson you can say what the vLLM scheduler does when the KV pool is full (it pauses a request, frees its blocks, and in V1 recomputes its context later), simulate a priority scheduler with preemption, compare recompute with a copy to CPU memory using numbers, and read the offload settings of the pinned image.

## Why it matters
- **Without a rule for a full pool:** the engine cannot admit the next request, or one long request keeps growing until nothing else fits. The queue stops draining.
- **Without a priority rule:** a nightly batch job that arrived first holds the blocks the interactive chat needs.
- **Without offload:** a customer who pauses for five minutes loses the KV cache of the conversation. The next message prefills the whole context again.
- **With them:** the scheduler pauses the lowest-priority request and frees its blocks. Offload keeps a copy of the KV in CPU memory or a shared store, so the return trip is a copy, not a recompute.
- **When you don't need it:** a pool with headroom for the peak load, or one user on a laptop. Then no request is ever paused.

## The idea
Think of a restaurant with a fixed number of tables. When the room is full and a VIP arrives, the host asks a walk-in to leave their table. The walk-in can leave their plates warm in the kitchen and come back to them (a **swap**: copy the KV to CPU memory and back), or order again from the start (**recompute**). vLLM's V1 engine always orders again: it frees the blocks and prefills the request's context a second time. **Offload** is the warming shelf: a copy of the KV kept in CPU RAM, or in a shared store, so a returning request is restored instead of recomputed.

## Picture
```mermaid
flowchart LR
  G["GPU KV pool: blocks in use"] -->|"pool full: lowest priority paused, blocks freed"| W["waiting queue"]
  W -->|"admitted again: prefill the context (recompute)"| G
  G -.->|"optional offload: copy to CPU RAM or a shared store"| C["CPU RAM, SSD or shared store"]
  C -.->|"restore the copy instead of recomputing"| G
```

## Step by step
### Step 1: what the pinned image says
The vLLM CPU image (`vllm/vllm-openai-cpu:v0.32.0`, the same tag as lessons 08, 11 and 12) contains the scheduler and the offload settings. The probe below imports them and prints the defaults, the policy values and the recompute line. It needs no model. The probe runs in Docker only when Docker is enabled here; otherwise the cell after it reads a saved copy.

In [1]:
LIBRARY_PROBE = """
import inspect, typing
from vllm.config import SchedulerConfig, CacheConfig
from vllm.v1.core.sched.scheduler import Scheduler
print('policy default:', SchedulerConfig(max_model_len=4096, is_encoder_decoder=False).policy)
print('policy values:', typing.get_args(typing.get_type_hints(SchedulerConfig)['policy']))
print('kv_offloading_size default:', CacheConfig().kv_offloading_size)
print('kv_offloading_backend default:', CacheConfig().kv_offloading_backend)
print('recompute reset:', [l.strip() for l in inspect.getsource(Scheduler).splitlines() if 'request.num_computed_tokens = 0' in l][0])
try:
    import lmcache
    print('lmcache: installed')
except ImportError:
    print('lmcache: not in this image (the native backend needs no extra package)')
"""

**Run the probe in the pinned image.** Docker starts the image's Python with `-I`, so the run ignores the working folder. This cell is skipped unless Docker is enabled.

In [2]:
import subprocess
from pathlib import Path

IMAGE = "vllm/vllm-openai-cpu:v0.32.0"           # the pinned CPU image (lessons 08, 11 and 12 use the same tag)
Path("../data/13").mkdir(parents=True, exist_ok=True)
run = subprocess.run(["docker", "run", "--rm", "--entrypoint", "python3", IMAGE, "-I", "-c", LIBRARY_PROBE],
                     capture_output=True, text=True, timeout=900)
if run.returncode == 0:
    Path("../data/13/library_probe.txt").write_text(run.stdout)
    print(run.stdout)
else:
    print("docker run failed:", run.stderr[-300:])

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**The saved output.** The probe above writes `data/13/library_probe.txt` when Docker runs. This cell reads that file if it exists. When Docker was skipped, there is no file, so the cell reads the copy in `data/recorded/13.json`, saved from an earlier run of the same image, and says so.

In [3]:
import json
from pathlib import Path
live = Path("../data/13/library_probe.txt")
if live.exists():
    print("saved output from the Docker run (data/13/library_probe.txt):")
    print(live.read_text())
else:
    recorded = json.loads(Path("../data/recorded/13.json").read_text())
    print("saved output from an earlier Docker run (" + recorded["image"] + "); Docker was skipped here:")
    print(recorded["stdout"])

saved output from the Docker run (data/13/library_probe.txt):
policy default: fcfs
policy values: ('fcfs', 'priority')
kv_offloading_size default: None
kv_offloading_backend default: native
recompute reset: request.num_computed_tokens = 0
lmcache: not in this image (the native backend needs no extra package)



**What you should see:** the default policy is `fcfs` (first come, first served), and the other value is `priority`. Offloading is off by default (`kv_offloading_size` is None) and the backend is `native`. The recompute line shows that a preempted request's computed-token count goes back to zero. The image does not contain `lmcache`, so the LMCache backend needs its package added before it can run.

## Under the hood
The scheduler lives inside the engine, so there is no separate library to call. Here is the mechanism in about 40 lines, in two cells. The scheduler works on blocks, not tokens: it admits a request when its context fits in the free blocks, and when a running request needs one more block and none is free, it pauses the lowest-priority request. Prefill time is not counted here. The point is the bookkeeping.

In [4]:
BLOCK = 16

def blocks(tokens):
    return -(-tokens // BLOCK)                        # ceiling: blocks for this many tokens of KV

def check_fits(prompts, outs, pool):
    """A request that needs more blocks than the whole pool can never run, even when it is alone."""
    worst = max(blocks(p + o) for p, o in zip(prompts, outs))
    if worst > pool:
        raise ValueError(f"a request needs {worst} blocks but the pool has {pool}: it can never run")

def admit(waiting, running, free, key):
    """Move waiting requests into the running set, best priority first, while their context fits."""
    for r in sorted(waiting, key=key):
        need = blocks(r["prompt"] + r["done"])
        if need <= free:
            free -= need; r["held"] = need
            running.append(r); waiting.remove(r)
    return free

**One step of the loop, in order.** A step is one pass in which every running request decodes one token.
1. **Admit.** Waiting requests join the running set, best priority first, while their context fits in the free blocks.
2. **Need a block?** Each running request, best priority first, may need one more block for its next token.
3. **Pause while the pool is full.** If no block is free, the inner `while` pauses the lowest-priority running request: it frees that request's blocks and puts it back in the queue. It repeats until the block fits. If the request that needs the block is itself the lowest priority, it is paused and the `break` ends the loop.
4. **Skip the paused.** A request paused earlier in this step is skipped.
5. **Recompute.** A paused request prefills all its prompt and generated tokens again when it is admitted. `redo` counts those tokens.

In [5]:
def simulate(prompts, outs, prios, pool):
    # Priority scheduler with recompute preemption. Each step, every running request decodes one token.
    check_fits(prompts, outs, pool)
    key = lambda r: (r["prio"], r["id"])              # lower value first, then arrival order
    reqs = [dict(id=i, prio=p, prompt=s, out=o, done=0, held=0) for i, (s, o, p) in enumerate(zip(prompts, outs, prios))]
    waiting, running = sorted(reqs, key=key), []
    free, preempted, redo, steps = pool, {0: 0, 1: 0}, 0, 0
    while any(r["done"] < r["out"] for r in reqs):
        steps += 1
        free = admit(waiting, running, free, key)
        for r in sorted(running, key=key):
            if r not in running:                      # paused earlier in this step
                continue
            extra = blocks(r["prompt"] + r["done"] + 1) - r["held"]   # blocks this token needs
            while extra > free:                       # pool full: pause the lowest priority
                v = max(running, key=key)
                running.remove(v); waiting.append(v)
                free += v["held"]; v["held"] = 0      # recompute: its blocks are gone
                preempted[v["prio"]] += 1; redo += v["prompt"] + v["done"]
                if v is r: break                      # r was the lowest: it waits, the step moves on
            if r in running:
                free -= extra; r["held"] += extra; r["done"] += 1
                if r["done"] == r["out"]:             # finished: its blocks go back at once
                    free += r["held"]; r["held"] = 0; running.remove(r)
    return steps, preempted, redo

### Step 2: the made-up trace and its priorities
The trace is the made-up support-desk trace from lesson 07: 30 requests, prompts of about 1,500 to 1,620 tokens, and 20 to 120 output tokens. It is generated with a fixed seed, not a real log. To give the scheduler something to choose between, this lesson labels every fourth request as a nightly batch job (priority 1, served last). The rest are interactive chats (priority 0). Lower numbers are served first, as in vLLM's `priority` policy.

In [6]:
import json
from pathlib import Path

trace = [json.loads(line) for line in Path("../data/07/support_trace.jsonl").read_text().splitlines()]
print(len(trace), "made-up requests; first three:")
for row in trace[:3]:
    print(row)

30 made-up requests; first three:
{'prompt_tokens': 1560, 'output_tokens': 24}
{'prompt_tokens': 1570, 'output_tokens': 45}
{'prompt_tokens': 1552, 'output_tokens': 33}


### Step 3: the same 30 requests, two policies, one pool
The pool has 400 blocks, which is 6,400 tokens of KV, in both runs. That is a toy size chosen so the trace overflows, not a real GPU's pool (lesson 05 computes the real one). The `fcfs` run gives every request the same priority, so it ignores the labels. Under `priority`, a batch job is paused whenever one is running. A chat is paused only when no batch job is running, and then the newest chat goes first. Each run prints the steps, how many times each priority was paused, and the tokens the engine had to prefill again.

In [7]:
prompts = [r["prompt_tokens"] for r in trace]
outs = [r["output_tokens"] for r in trace]
prios = [1 if i % 4 == 3 else 0 for i in range(len(trace))]     # made-up: every fourth request is a batch job

print(f"{'policy':<10}{'steps':>7}{'paused: chat':>14}{'batch':>7}{'recomputed tokens':>20}")
for name, p in (("fcfs", [0] * len(trace)), ("priority", prios)):
    steps, pre, redo = simulate(prompts, outs, p, pool=400)
    print(f"{name:<10}{steps:>7}{pre[0]:>14}{pre[1]:>7}{redo:>20,}")

policy      steps  paused: chat  batch   recomputed tokens
fcfs          635            79      0             122,309
priority      628            11      4              23,229


**What you should see:** with `fcfs` the newest running request is the one paused, again and again, and the engine recomputes about 122,000 tokens. With `priority` the batch jobs are paused first, the chats are paused about 11 times instead of 79, and the recomputed work falls to about 23,000 tokens. The batch count is small here, and it can rise when the pool is bigger: more batch jobs are admitted, and each one can then be paused. These counts come from the simulator, not from a GPU. Trust the ranking, not the exact numbers.

### Step 4: swap or recompute, with numbers
A paused request can be restored in two ways. A **swap** copies its KV blocks to CPU memory and back, which costs PCIe time. A **recompute** prefills its context again, which costs GPU time. The table uses the KV size of Llama 3.1 8B in FP16 (128 KiB per token, lesson 03) and the one-way PCIe speed of an L4 (about 32 GB/s per direction: NVIDIA lists 64 GB/s total for the L4's PCIe Gen4 x16, shared by both directions). The prefill speed is an **assumption**: 0.25 ms per token, about 4,000 prompt tokens per second. Replace it with your own TTFT measurement from lesson 07.

In [8]:
KV_BYTES_PER_TOKEN = 128 * 1024        # Llama 3.1 8B, FP16 (lesson 03)
PCIE_BYTES_PER_S = 32e9                # one direction; NVIDIA lists 64 GB/s total for the L4's PCIe Gen4 x16
PREFILL_MS_PER_TOKEN = 0.25            # ASSUMED for this table, not measured

print(f"{'context':>8}{'swap out + in (ms)':>20}{'recompute (ms)':>16}")
for ctx in (200, 1500, 4000):
    swap = 2 * ctx * KV_BYTES_PER_TOKEN / PCIE_BYTES_PER_S * 1000
    redo = ctx * PREFILL_MS_PER_TOKEN
    print(f"{ctx:>8}{swap:>20.1f}{redo:>16.0f}")

 context  swap out + in (ms)  recompute (ms)
     200                 1.6              50
    1500                12.3             375
    4000                32.8            1000


On paper, copying is cheaper at every length in the table, and the gap grows with the context. vLLM V1 still recomputes by default. The reasons follow. A CPU buffer has to be sized and managed. The copy shares the PCIe link with other traffic. And with prefix caching (lesson 14, next: reusing KV blocks already computed for the same text), a paused request's blocks often stay in the cache, so its re-prefill is partly a cache hit. Choose with a measured prefill speed, not with the assumed one.

### Step 5: the offload settings
Offload is off until you set a size in GiB (with tensor parallelism, the total over all ranks). The `native` backend is vLLM's own CPU buffer and needs no extra package. These are GPU-host settings, so this lesson shows them without running them.

```bash
vllm serve <model> --kv-offloading-size 8 --kv-offloading-backend native
```

The `lmcache` backend uses the LMCache layer instead, and it needs its package installed.

**The 525% figure.** Mooncake, the KV store behind Kimi, pools CPU, DRAM and SSD across machines. Its paper (arXiv 2407.00079) reports up to 525% more throughput than a baseline method in simulated scenarios. The abstract does not name that baseline, so read the number as the authors' simulation result, not a production gain. A copy is worth keeping only if it comes back.

## At work
- **Preemption storms are a common production failure.** The counter rises and TPOT p95 rises while the load stays flat, because each pause repeats prefill work. Watch the counter next to the queue length. → lesson 33 (reliability drill).
- **Cache hits are the cost lever.** DeepSeek reported that 56.3% of its input tokens hit the on-disk KV cache over 24 hours (February 2025). Offload pays only when the hit rate is high. → lesson 14 (prefix caching), lesson 34 (multi-node).
- **A cache that survives a crash.** LMCache can run as a standalone daemon, so the KV survives an engine crash. Measure the restore against a full re-prefill before you rely on it. → lesson 34 (multi-node serving).

Sources: vLLM docs (docs.vllm.ai); DeepSeek open-infra-index, day 6 (github.com/deepseek-ai/open-infra-index); LMCache (github.com/LMCache/LMCache).

## Common mistakes
**1. Sending a request that cannot fit the pool even when it is alone.** A request that needs more blocks than the whole pool can never run, because preempting everything else would not make room for it. The simulator refuses it, and a real engine has the same limit at its maximum context length.

In [9]:
simulate([1500], [60], [0], pool=50)     # ValueError: a request needs 98 blocks but the pool has 50

ValueError: a request needs 98 blocks but the pool has 50: it can never run

**2. Setting priorities and leaving the policy at `fcfs`.** The request's priority is ignored unless the server runs with the priority policy. Start the server with the flag, then check the policy in the startup log.

```bash
vllm serve <model> --scheduling-policy priority
```

**3. Reading a rising preemption count as harmless.** Every pause repeats prefill work, so the count is wasted GPU time. Fix the cause (pool size, `max_num_seqs`, load), not the symptom.

## Try it
Predict first: with `priority`, if the pool doubles from 400 to 800 blocks, will the recomputed tokens fall by about half, by much more, or by less? Run the cell with 800 blocks and compare the number with the Step 3 result under `priority`.

In [10]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [11]:
for pool in (400, 800):
    steps, pre, redo = simulate(prompts, outs, prios, pool=pool)
    print(f"pool {pool}: paused {pre}, recomputed {redo:,} tokens")
# With 800 blocks the recomputed work is about 47% of the 400-block value (10,915 against 23,229), a little less than half.
# The chat pauses fall from 11 to 2, but the batch pauses rise from 4 to 5: a bigger pool admits more batch jobs, and each one can then be paused.

pool 400: paused {0: 11, 1: 4}, recomputed 23,229 tokens
pool 800: paused {0: 2, 1: 5}, recomputed 10,915 tokens


## Recap
- When the KV pool is full, the scheduler pauses a running request and frees its blocks. In vLLM V1 the paused request's context is recomputed when it is admitted again.
- The `priority` policy pauses the lowest-priority request first; `fcfs` pauses the newest. Priorities work only under `priority`.
- A swap copies KV to CPU memory and back instead. Offload keeps the copy in CPU RAM or a shared store. Both trade PCIe time for GPU time; measure your own prefill speed before choosing.
- A rising preemption count means wasted prefill work. Watch it next to the queue.

**Interview:** "What happens in vLLM when the KV cache is full?" Strong answer: the scheduler pauses the lowest-priority running request, frees its blocks and requeues it; V1 recomputes its context on re-admission, so each pause costs prefill work. Offload trades that work for copies. The fixes are a bigger pool, less load, and admission control.